# Day 14: RAG Integration

## Goal

Take extracted PDF text, store it in a vector database, and ask questions over that content.

By the end of this notebook you will have:

- extracted raw text from a PDF
- split that text into chunks
- stored those chunks in Chroma
- retrieved relevant chunks for a question
- generated an answer from the retrieved context

This is where document extraction starts becoming a usable Q&A pipeline.

## Step 0: Sync the Project Once

The Day 14 packages are already included in `pyproject.toml`, so learners only need:

```bash
uv sync
```

Set the values you want in `.env`.

Example:

```text
OPENAI_API_KEY=your_openai_key
OPENAI_MODEL=your_openai_model
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
OLLAMA_MODEL=llama3.2
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_EMBEDDING_MODEL=nomic-embed-text
```

This notebook supports either OpenAI or Ollama for both chat and embeddings.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pypdf import PdfReader

print("Imports loaded successfully.")

In [ ]:
load_dotenv(override=True)

config_summary = {
    "OPENAI_API_KEY_present": bool(os.getenv("OPENAI_API_KEY")),
    "OPENAI_MODEL": os.getenv("OPENAI_MODEL"),
    "OPENAI_EMBEDDING_MODEL": os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    "OLLAMA_MODEL": os.getenv("OLLAMA_MODEL"),
    "OLLAMA_BASE_URL": os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
    "OLLAMA_EMBEDDING_MODEL": os.getenv("OLLAMA_EMBEDDING_MODEL"),
}

config_summary

## Step 1: Extract Text from a PDF

We use the sample PDF already in the repository.

If you want to test invoice processing later, replace this file with an invoice PDF or use OCR output from Day 13.

In [ ]:
pdf_path = Path("resources/AI Hands-on Training.pdf")
pdf_path.exists(), pdf_path

In [ ]:
def extract_text_from_pdf(pdf_path: Path, max_pages: int | None = None) -> str:
    reader = PdfReader(pdf_path)
    pages = reader.pages if max_pages is None else reader.pages[:max_pages]
    chunks = []
    for page in pages:
        chunks.append(page.extract_text() or "")
    return "\n\n".join(chunks)


raw_text = extract_text_from_pdf(pdf_path, max_pages=8)
print(raw_text[:3000])

## Step 2: Chunk the Extracted Text

Vector databases work better with smaller text chunks than with one giant document.

For Day 14, we use a simple character-based splitter to keep the logic easy to understand.

In [ ]:
def split_text(text: str, chunk_size: int = 800, overlap: int = 120) -> list[str]:
    pieces = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        pieces.append(text[start:end])
        start += chunk_size - overlap
    return [piece.strip() for piece in pieces if piece.strip()]


text_chunks = split_text(raw_text)
len(text_chunks)

In [ ]:
text_chunks[0][:1500]

In [ ]:
documents = [
    Document(
        page_content=chunk,
        metadata={"source": str(pdf_path), "chunk_id": index},
    )
    for index, chunk in enumerate(text_chunks)
]

documents[:2]

## Step 3: Create Helpers for Chat and Embeddings

This notebook supports either OpenAI or Ollama.

In [ ]:
def get_chat_model(provider: str):
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        model = os.getenv("OPENAI_MODEL")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is not set in .env")
        if not model:
            raise ValueError("OPENAI_MODEL is not set in .env")
        return ChatOpenAI(model=model, api_key=api_key)

    if provider == "ollama":
        model = os.getenv("OLLAMA_MODEL")
        if not model:
            raise ValueError("OLLAMA_MODEL is not set in .env")
        return ChatOllama(
            model=model,
            base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
        )

    raise ValueError("provider must be 'openai' or 'ollama'")


def get_embedding_model(provider: str):
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        model = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is not set in .env")
        return OpenAIEmbeddings(model=model, api_key=api_key)

    if provider == "ollama":
        model = os.getenv("OLLAMA_EMBEDDING_MODEL")
        if not model:
            raise ValueError("OLLAMA_EMBEDDING_MODEL is not set in .env")
        return OllamaEmbeddings(
            model=model,
            base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
        )

    raise ValueError("provider must be 'openai' or 'ollama'")


print("Provider helpers ready.")

## Step 4: Store the Chunks in Chroma

This turns the extracted PDF text into a searchable vector store.

In [ ]:
provider = "openai"  # Change to "ollama" if you want to use Ollama.

embedding_model = get_embedding_model(provider)
db_path = Path("day14/chroma_store")
db_path.mkdir(parents=True, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=str(db_path),
    collection_name="day14_pdf_chunks",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector store and retriever ready.")

## Step 5: Test Retrieval First

Always inspect the retrieved chunks before you trust the generated answer.

In [ ]:
question = "What topics are covered in the training material?"
question

In [ ]:
retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"Chunk {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:1200])
    print("-" * 80)

## Step 6: Build the RAG Chain

Now we combine:

- retrieval from Chroma
- prompt templating
- the chat model
- a text output parser

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chat_model = get_chat_model(provider)

rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.
Answer the user's question using only the context below.
If the answer is not clearly supported by the context, say you do not know.

Context:
{context}

Question:
{question}
    """.strip()
)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | chat_model
    | StrOutputParser()
)

print("RAG chain ready.")

In [ ]:
rag_chain.invoke(question)

## Step 7: Ask Another Question

Try changing the question below based on the extracted material.

Examples:

- `What is the training focused on?`
- `What kinds of exercises are described?`
- `How does the material describe the purpose of the course?`

If you later replace the source PDF with an invoice, the same pipeline can answer invoice-related questions.

In [ ]:
rag_chain.invoke("What is the purpose of the training material?")

## Day 14 Recap

You now have an end-to-end document Q&A flow:

- extract text from a PDF
- split the text into chunks
- store the chunks in Chroma
- retrieve relevant chunks for a question
- generate an answer from the retrieved context

This is the key integration step between document extraction and usable RAG.